In [1]:
%load_ext cudf.pandas

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
X_train = pd.read_csv('data/train.csv')
Y_train = pd.read_csv('data/train_labels.csv')
target_pairs = pd.read_csv('data/target_pairs.csv')

In [6]:
print(X_train.shape)
print(Y_train.shape)
print(target_pairs.shape)


(1961, 558)
(1961, 425)
(424, 3)


In [12]:
fx_pairs = [(col[3:6], col[6:9]) for col in X_train.columns if 'FX_' in col]
fx_set = set(elem for pair in fx_pairs for elem in pair)
fx_set

{'AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NOK', 'NZD', 'USD', 'ZAR'}

In [29]:
from collections import defaultdict

categories = {
    "Open": "open",
    "adj_open": "open",
    "Close": "close",
    "adj_close": "close",
    "High": "high",
    "adj_high": "high",
    "Low": "low",
    "adj_low": "low",
    "open_interest": "open_interest",
    "Volume": "volume",
    "adj_volume": "volume",
    "settlement_price": "settlement_price",
}
by_stem = defaultdict(set)
for col in X_train.columns:
    if "FX" in col or col == "date_id":
        continue
    entry, curr_cat = next(((cat, categories[cat]) for cat in categories if cat in col), (None, None))
    if not entry:
        print("can't parse", col)
    stem = col.replace(entry, "").strip("_")
    by_stem[stem].add(curr_cat)

for stem, cats in by_stem.items():
    print(stem, cats)
       

LME_AH {'close'}
LME_CA {'close'}
LME_PB {'close'}
LME_ZS {'close'}
JPX_Gold_Mini_Futures {'settlement_price', 'volume', 'low', 'open', 'close', 'open_interest', 'high'}
JPX_Gold_Rolling-Spot_Futures {'settlement_price', 'volume', 'low', 'open', 'close', 'open_interest', 'high'}
JPX_Gold_Standard_Futures {'volume', 'low', 'open', 'close', 'open_interest', 'high'}
JPX_Platinum_Mini_Futures {'settlement_price', 'volume', 'low', 'open', 'close', 'open_interest', 'high'}
JPX_Platinum_Standard_Futures {'volume', 'low', 'open', 'close', 'open_interest', 'high'}
JPX_RSS3_Rubber_Futures {'settlement_price', 'volume', 'low', 'open', 'close', 'open_interest', 'high'}
US_Stock_ACWI {'volume', 'low', 'open', 'close', 'high'}
US_Stock_AEM {'volume', 'low', 'open', 'close', 'high'}
US_Stock_AG {'volume', 'low', 'open', 'close', 'high'}
US_Stock_AGG {'volume', 'low', 'open', 'close', 'high'}
US_Stock_ALB {'volume', 'low', 'open', 'close', 'high'}
US_Stock_AMP {'volume', 'low', 'open', 'close', 'high'

In [52]:
print(target_pairs[["lag", "pair"]].to_string(max_rows=None))

     lag                                                           pair
0      1                                          US_Stock_VT_adj_close
1      1                           LME_PB_Close - US_Stock_VT_adj_close
2      1                                    LME_CA_Close - LME_ZS_Close
3      1                                    LME_AH_Close - LME_ZS_Close
4      1                 LME_AH_Close - JPX_Gold_Standard_Futures_Close
5      1             LME_ZS_Close - JPX_Platinum_Standard_Futures_Close
6      1                                    LME_PB_Close - LME_AH_Close
7      1                          LME_ZS_Close - US_Stock_VYM_adj_close
8      1      US_Stock_IEMG_adj_close - JPX_Gold_Standard_Futures_Close
9      1                                       FX_AUDJPY - LME_PB_Close
10     1  JPX_Platinum_Standard_Futures_Close - US_Stock_VGIT_adj_close
11     1                                       FX_EURAUD - LME_CA_Close
12     1                JPX_Platinum_Standard_Futures_Close - FX

In [55]:
np.log(X_train["LME_PB_Close"] / X_train["US_Stock_VT_adj_close"]).diff()[-5:]

1956    0.019426
1957   -0.000225
1958   -0.004500
1959   -0.001036
1960   -0.002032
dtype: float64

In [56]:
Y_train["target_1"][-5:]

1956   -0.004500
1957   -0.001036
1958   -0.002032
1959   -0.006335
1960    0.006502
Name: target_1, dtype: float64